# Skyra TSER Recon

### 1. Import Libraries

In [1]:
import mapvbvd
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib qt

/home/hans/Documents/TSER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Load dataset

In [80]:
def load_TSER(file, points, echoes, channels = 15):
    twixObj = mapvbvd.mapVBVD(file)
    twixObj[1].image.squeeze = True #remove extra dimensions
    twixObj[1].image.flagRemoveOS = False #no readout oversampling
    data = twixObj[1].image[''].reshape(points,channels,-1, 2, echoes) # arrange data into (points, channels, slices, phase cycle, echoes)
    data_pc = data[:,:,:,0] + data[:,:,:,1] # add phase cycle steps to remove ringing

    # data_pc is phase cycled data with shape (points, channels, slices, echoes)
    return data_pc

data = load_TSER('Data/CMIF/Oct5/meas_MID00101_FID02502_400a_std_200.dat', 128, 400)

pymapVBVD version 0.7.0
Software version: VD


### 3. Fourier Transform to get Image

In [81]:
data_ft = np.fft.fftshift(np.fft.fft(np.fft.fftshift(data[3:-3,:,:,:], axes=(0)), axis=0), axes=(0))

### 4. Plot 3D data for one slice

In [84]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.view_init(elev=30, azim=-26, roll=0)

distance = np.linspace(-54,54,np.shape(data_ft)[0])
colors = plt.cm.viridis(np.linspace(0, 1, 400))

for i in range(400):
    ax.plot(i, distance, np.abs(data_ft[:,0,0,i]),color=colors[i], lw=1) #every point, channel 6, slice 0, echo 'i'

plt.show()

### 5. Plot Slice Cross Section

In [141]:
%matplotlib qt
plt.figure()
plt.plot(np.abs(data_ft[:,0,0,5])) #Every point, channel 6, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 5.5 Plot Plate Map

In [146]:
fig, ax = plt.subplots(1,1)
ax.imshow(np.abs(np.fliplr(data_ft[:,0,:,5])), aspect='auto') #Every point, channel 6, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 6. Plot Decay for One Point

In [154]:
point = 21
echoes = 400
te = 10e-3

def get_t2(echo_data, te):
    def _decay(x, a, b, c):
        return a * np.exp(-b * x)+c
    popt, _ = opt.curve_fit(_decay, np.arange(len(echo_data))*te, echo_data/np.max(echo_data))
    return 1/popt[1]

plt.figure()
time = np.arange(echoes)*te
plt.plot(time, np.abs(data_ft[24,3,0,:])) #Point 40, channel 6, slice 0, every echo
t2 = get_t2(np.abs(data_ft[24,4,0,:]), te)
print(t2)
plt.xlabel("Distance (mm)")
plt.ylabel("Amplitude")
# plt.legend("%2.2f" % t2)
plt.show()

2.336131089962503


In [ ]:
import scipy.optimize as opt


In [173]:
concentrations = np.array([2.5, 
                           25, 
                           100, 
                           10, 
                           5, 
                           100, 
                           50, 
                           5, 
                           10, 
                           1, 
                           0, 
                           2.5])
                           
t2s = np.array([2.0484010298621986, 
                0.9412785614114619, 
                0.29863239816674403, 
                1.4923054411404348, 
                1.8040829106928467, 
                0.32854632286652946,
                0.5639687423136152,
                1.8569891066640016,
                1.5336809002803322,
                2.336131089962503,
                2.6332997921143266,
                2.1271393464235238
                ])

conc_mit = np.array([0.015625,
0.0078125,
0.00390625,
0.001953125,
0.0009765625,
0])

t2_mit = np.array([0.7549,
1.1312,
1.5865,
1.9365,
2.1923,
2.4922])

larmor_conc = np.array([1000,500,100,50,10,5,1,0.8,0.5,0.25,0.1,0])
larmor_t2 = np.array([0.065,0.126,0.528,0.879,1.865,2.185,2.495,2.529,2.549,2.569,2.565,2.581])

tn = plt.figure()
plt.plot(conc_mit*1000, 1/t2_mit, '.')
plt.plot(concentrations, 1/t2s, '.')
plt.plot(larmor_conc, 1/larmor_t2, '.')
# plt.xscale('log')
# plt.yscale('log')
plt.xlabel('Concentration (uM)')
plt.ylabel('$R_2$')
plt.legend(['PMIT 7T', 'CMIF 3T', 'Larmor 0.5T'])
# plt.xlim([-1, 18])
# plt.ylim([-1, 18])
plt.grid()
plt.show()